# Cuál es el problema?

Abandono de clientes (Churn)

Trabajamos con una empresa de telecomunicaciones que quiere ***predecir cuales clientes van a abandonar el servicio*** (churn) para poder tomar acciones preventivas.

Por qué es importante?

* Conseguir un cliente nuevo representa entre 5x y 25x más que retener uno existente.
* Si podemos predecir quien se va, podemos ofrecerle un descuento o llamarlo antes de que cancele.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer # Para transformar columnas específicas
from sklearn.preprocessing import OneHotEncoder # Para escalar y codificar variables
from sklearn.linear_model import LogisticRegression # Modelo de regresión logística

# Metricas de evaluación para la clasificación

from sklearn.metrics import (
    confusion_matrix, # Matriz de confusión
    classification_report, # reporte con precisión, recall y f1-score
    accuracy_score, # Porcentaje de predicciones correctas
)

import matplotlib.pyplot as plt

In [ ]:
#Cargar el archivo CSV en un DataFrame (tabla de datos  )
df = pd.read_csv("abandono_clientes.csv")
# Mostrar las primeras filas del DataFrame para verificar que se cargó correctamente
print("primeras 5 filas del DataSet:")
df.head()

In [ ]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

aqui viene algo

In [ ]:
#Comporar el promedio de cada variable 

comparacion = df.groupby("abandono")[["antiguedad_meses","factura_mensual",
                                      "llamadas_soporte","satisfaccion"]].mean().round(2)
comparacion.index=["Permanece (0)","Abandona (1)"]
print("Promedio de variables numerica segun abandono:")
comparacion

In [ ]:
# Analisi del tipo contrato vs el abandono
tabla_contrato = pd.crosstab(
    df["contrato"], 
    df["abandono"], 
    normalize="index"
    ) * 100
tabla_contrato.columns = ["Permanece (%)", "Abandona (%)"]
print("porcentaje de abandono pro tipo de contrato:")
comparacion

# preparar los daros para el modelo

1. separar variables predictoras (x) de la variable objetivo
2. Entrenar los datos con (80%) del dataset y probar con el (20%) del dataset.

In [ ]:
# paso 1: separar x (variables predictoras) e y (variable objetivo)
X = df[["antiguedad_meses", "factura_mensual", "llamadas_soporte", "satisfaccion", "contrato"]]
y = df["abandono"]

# paso 2: dividir el conjunto de entrenamiento (80%) y prueba (20% )
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, #20% para prueba
        random_state=42,  # Semilla para reproducibilidad
    stratify=y  
)

print(f"Datos de entrenamiento: {X_train.shape[0]} clientes")
print(f"Datos de prueba: {X_test.shape[0]} clientes")
print("")
print(f"Proporción de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporción de abandono en prueba: {y_test.mean()*100:.1f}%")
print()
print("Las proporciones son similares gracias al parametro 'stratify=y' en train_test_split")


# Construir  el pipeline y entrenar el modelo

1. Calula una puntuación lineal
2.Pasa por una puntación que la convierte en probabilidad entre 0 y 1
3. SI la probabilidad es mayor a 0.5  predice la clase 1 (abandona): si no, predice 0 (permanece)

In [ ]:
from sklearn.pipeline import Pipeline

numeric_features = ["antiguedad_meses", "factura_mensual",
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]


preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)


pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

# Entrenar el modelo
pipe.fit(X_train, y_train)

print("Modelo entrenado con éxito.")

# Hacer Predcciones

Usamos el modelo entrenado para predecir el abandono en los datos de prueba

In [ ]:
# Generar predicciones con los datos de prueba
y_pred = pipe.predict(X_test)

# Mostrar las primeras 10 predicciones vs valores reales
comparacion_pred = pd.DataFrame({
    "Real":      y_test.values[:10],
    "Prediccion": y_pred[:10],
    "Correcto":  ["Si" if r == p else "NO" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("Primeras 10 predicciones vs valores reales:")
print()
comparacion_pred

# Evaluacion del modelo: Exactitu/precision (Acurracy)

cuando hablamos del porcentaje de precision nos referimos al % de predicciones que fueron correctas

Accuracy = Predicciones Correctas / Total de predicciones

Ejemplo: si el 95% de clientes **No** abandona, un modelo que siempre diga "No abandona" tendría un 95% de presicion, pero jamas detectaria a un cliente en riesgo. sería inutil.

In [ ]:
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy (exactitud): {acc:.4f} ({acc*100:.1f}%%)")
print()
print(f"esto significa que el modelo acerto {int(acc*len(y_test))} de {len(y_test)} predicciones")


In [ ]:
# Calcular la matriz de confusión
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Matriz de Confusión:")
print(f"  Verdaderos Negativos (TN): {tn} — permanece y el modelo dijo permanece")
print(f"  Falsos Positivos    (FP): {fp} — permanece pero el modelo dijo abandona")
print(f"  Falsos Negativos    (FN): {fn} — abandona pero el modelo dijo permanece")
print(f"  Verdaderos Positivos(TP): {tp} — abandona y el modelo dijo abandona")
print()
print(f"Errores totales: {fp + fn} de {len(y_test)} predicciones")

In [ ]:
# Reporte de clasificación completo
print("Reporte de Clasificación:")
print()
print(classification_report(y_test, y_pred, 
                            target_names=["Permanece (0)", "Abandona (1)"]))

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfica 1: Matriz de Confusión como heatmap ---
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Permanece", "Abandona"],
            yticklabels=["Permanece", "Abandona"],
            ax=axes[0], cbar=False,
            annot_kws={"size": 16})
axes[0].set_xlabel("Predicción del modelo", fontsize=12)
axes[0].set_ylabel("Valor real", fontsize=12)
axes[0].set_title("Matriz de Confusión", fontsize=14, fontweight="bold")

# --- Gráfica 2: Importancia de las variables (coeficientes del modelo) ---
# Obtener nombres de las variables después del preprocesamiento
cat_names = pipe.named_steps["preprocess"] \
    .transformers_[1][1].get_feature_names_out(["contrato"]).tolist()
feature_names = numeric_features + cat_names

coefs = pipe.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({"Variable": feature_names, "Coeficiente": coefs})
coef_df = coef_df.sort_values("Coeficiente")

colors = ["#2ecc71" if c < 0 else "#e74c3c" for c in coef_df["Coeficiente"]]
axes[1].barh(coef_df["Variable"], coef_df["Coeficiente"], color=colors)
axes[1].set_xlabel("Coeficiente (peso en el modelo)", fontsize=12)
axes[1].set_title("Importancia de Variables", fontsize=14, fontweight="bold")
axes[1].axvline(x=0, color="gray", linestyle="--", linewidth=0.8)

# Leyenda
axes[1].text(0.98, 0.02, "Rojo = aumenta abandono\nVerde = reduce abandono",
             transform=axes[1].transAxes, fontsize=9,
             ha="right", va="bottom",
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()